In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestClassifier

In [2]:
# Definindo os caminhos - notebook em scripts/Redução de Dimensionalidade/
caminho_pasta_tratado = '../../dataset tratado/cicids2017/'

nome_dados_treinamento = 'Sem Redução de Dimensionalidade/cicids2017_treinamento.csv'
nome_dados_teste = 'Sem Redução de Dimensionalidade/cicids2017_teste.csv'

nome_dados_treinamento_reduzidos = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_balanceados.csv'
nome_dados_teste_reduzidos = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos_balanceados.csv'

In [3]:
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")
display(df_treino.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1979513, 81) | Teste: (848363, 81)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.006760,0.855604,0.352941,3.229166e-05,0.000009,0.000003,2.945736e-06,9.153974e-09,0.001531,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.625956,0.006760,0.352941,2.272266e-03,0.000009,0.000017,0.000000e+00,4.576987e-09,0.000000,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.006760,0.848402,0.352941,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.801816,0.000809,1.000000,1.233333e-06,0.000005,0.000007,6.821705e-06,1.830795e-07,0.001773,0.018925,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.583032,0.000809,1.000000,1.449567e-03,0.000005,0.000007,4.496124e-06,3.509023e-07,0.001168,0.012473,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [4]:
# 2. Removendo características identificadoras/enviesadas antes da seleção por MDI
nomes_enviesados = {
    'Flow ID', 'Source IP', 'Source Port', 'Destination IP',
    'Destination Port', 'Protocol', 'Timestamp', 'Fwd Header Length.1'
}

colunas_para_remover = []
fwd_header_length_vistos = 0

for coluna in df_treino.columns:
    nome_normalizado = coluna.strip().lower()

    if coluna in nomes_enviesados:
        colunas_para_remover.append(coluna)

    if nome_normalizado == 'fwd header length':
        fwd_header_length_vistos += 1
        if fwd_header_length_vistos > 1:
            colunas_para_remover.append(coluna)

    if nome_normalizado.startswith('fwd header length.'):
        colunas_para_remover.append(coluna)

colunas_para_remover = [c for c in dict.fromkeys(colunas_para_remover) if c != 'Label']
colunas_treino_existentes = [c for c in colunas_para_remover if c in df_treino.columns]
colunas_teste_existentes  = [c for c in colunas_para_remover if c in df_teste.columns]

df_treino = df_treino.drop(columns=colunas_treino_existentes)
df_teste  = df_teste.drop(columns=colunas_teste_existentes)

print('Características removidas para mitigar overfitting:', colunas_treino_existentes)
print(f'Dimensões após remoção - Treino: {df_treino.shape} | Teste: {df_teste.shape}')

# 3. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

Características removidas para mitigar overfitting: ['Source Port', 'Destination Port', 'Protocol', 'Fwd Header Length.1']
Dimensões após remoção - Treino: (1979513, 77) | Teste: (848363, 77)


In [5]:
# 4. Aplicando Seleção de Características MDI com Random Forest Balanceado
print("\nIniciando o treinamento do Random Forest para extração de features (MDI)...")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced')

inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()

print(f"Treinamento RF concluído! Tempo extração: {(fim_fs - inicio_fs):.4f} segundos.")

importancias = rf_fs.feature_importances_
features = X_treino.columns

df_importancias = pd.DataFrame({'Feature': features, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)

print("\nQuantidade total de features:", len(features))

# Selecionando as 51 features mais importantes (conforme sugerido na literatura por Neto e Gomes)
top_51_features = df_importancias.head(51)['Feature'].tolist()

quantidade_de_features_descartadas = len(features) - len(top_51_features)
print("Quantidade de features selecionadas (top 51):", len(top_51_features))
print("Quantidade de features descartadas:", quantidade_de_features_descartadas)

importancia_corte = df_importancias[df_importancias['Feature'] == top_51_features[-1]]['Importancia'].values[0]
print(f"Importância mínima para entrar no top 51: {importancia_corte:.6f}")
display(df_importancias.tail(quantidade_de_features_descartadas + 5))

print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))


Iniciando o treinamento do Random Forest para extração de features (MDI)...
Treinamento RF concluído! Tempo extração: 57.7522 segundos.

Quantidade total de features: 76
Quantidade de features selecionadas (top 51): 51
Quantidade de features descartadas: 25
Importância mínima para entrar no top 51: 0.005304


,Feature,Importancia
75,Idle Min,6.452902e-03
47,URG Flag Count,6.078209e-03
26,Bwd IAT Std,5.663514e-03
28,Bwd IAT Min,5.454974e-03
68,Active Mean,5.304087e-03
46,ACK Flag Count,5.183622e-03
50,Down/Up Ratio,4.827728e-03
6,Fwd Packet Length Min,3.836841e-03
34,Bwd Header Length,2.691147e-03
4,Total Length of Bwd Packets,2.672074e-03



Top 10 features mais importantes:


,Feature,Importancia
65,Init_Win_bytes_backward,0.084826
64,Init_Win_bytes_forward,0.041466
53,Avg Bwd Segment Size,0.033042
38,Max Packet Length,0.032917
5,Fwd Packet Length Max,0.031299
51,Average Packet Size,0.030972
15,Flow IAT Mean,0.030597
16,Flow IAT Std,0.030563
19,Fwd IAT Total,0.028077
11,Bwd Packet Length Mean,0.027105


In [6]:
# 5. Reduzindo a dimensionalidade dos conjuntos de treino e teste
df_treino_reduzido = X_treino[top_51_features].copy()
df_treino_reduzido['Label'] = y_treino.values

df_teste_reduzido = X_teste[top_51_features].copy()
df_teste_reduzido['Label'] = y_teste.values

print(f"Novas dimensões - Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")

# Salvando os datasets reduzidos em CSV para uso posterior
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_dados_treinamento_reduzidos, index=False)
print(f"Dataset de treino reduzido salvo em: {caminho_pasta_tratado + nome_dados_treinamento_reduzidos}")

df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_dados_teste_reduzidos, index=False)
print(f"Dataset de teste reduzido salvo em: {caminho_pasta_tratado + nome_dados_teste_reduzidos}")

Novas dimensões - Treino: (1979513, 52) | Teste: (848363, 52)
Dataset de treino reduzido salvo em: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_balanceados.csv
Dataset de teste reduzido salvo em: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos_balanceados.csv
